# Sentinel-3A OLCI Extraction

This notebook demonstrates how to search for Sentinel-3A OLCI granules via **earthaccess** and extract ocean color bands using **satpy**.

Sentinel-3 products ship as `.zip` archives — the extractor handles automatic expansion.

## Setup

Import the required libraries and configure the search parameters.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import time

import geopandas as gpd

from aer.client import AerClient
from aer.interfaces import ExtractionProfile
from aer.search_earthaccess import earthaccess_download_wrapper

## Configuration

Define the AOI, time range, and collection.

In [ ]:
# --- Configuration ---
DATE_START = datetime(2026, 4, 1, 0, 0, tzinfo=timezone.utc)
DATE_END = datetime(2026, 4, 2, 0, 0, tzinfo=timezone.utc)

# Load AOI (Córdoba, Argentina)
geojson_path = Path("cordoba.geojson")
gdf = gpd.read_file(geojson_path)
aoi = gdf.geometry.iloc[0]

# Sentinel-3A OLCI L1B EFR collection
collections = ["S3A_OL_1_EFR"]

# --- Client Setup ---
client = AerClient()

## Search

Search for Sentinel-3 OLCI granules via **earthaccess**.

In [ ]:
print("Searching...", flush=True)
results = client.search(
    collections=collections,
    start_datetime=DATE_START,
    end_datetime=DATE_END,
    intersects=aoi,
    plugin_hints={"search_earthaccess": collections},
)
print(f"Found {len(results)} results", flush=True)
results.head()

## Prepare Extraction

Define extraction profiles and prepare tasks.

In [ ]:
# --- Prepare Extraction ---
uri = "extract_test_sentinel3"

profiles = [
    ExtractionProfile(
        name="olci_rgb",
        resolution=300,
        collection_variables_map={"S3A_OL_1_EFR": ["Oa04"]},
        extra_params={"reader": "olci_l1b"},
    )
]

prepare_params = {"cells_per_chunk": 10}

tasks = client.prepare_for_extraction(
    results,
    target_aoi=aoi,
    uri=uri,
    profiles=profiles,
    target_grid_dist=256000,
    target_grid_overlap=False,
    prepare_params=prepare_params,
    plugin_hints={"extract_satpy": collections},
)

print(f"Prepared {len(tasks)} extraction tasks", flush=True)

## Extract

Run the extraction pipeline.

> Note: Sentinel-3 products are delivered as `.zip` archives. The `extract_satpy` plugin will automatically expand them and discover the `.SEN3` directory contents.

In [ ]:
# Clean output directory
uri_path = Path(uri)
if uri_path.exists():
    shutil.rmtree(uri_path)
uri_path.mkdir(parents=True)

print("Extracting...", flush=True)
start_time = time.time()

extract_params = {
    "downloader": earthaccess_download_wrapper,
    "calibration": "reflectance",
    "padding": 2,
    "resampler": "nearest",
}

results_df = client.extract_batches(
    tasks,
    extract_params=extract_params,
    plugin_hints={"extract_satpy": collections},
    max_batch_workers=1,
)

end_time = time.time()
print(f"Extraction took {end_time - start_time:.2f}s")
print(f"Extracted {len(results_df)} artifacts")

## Results

Inspect the extracted artifacts.

In [ ]:
results_df[["id", "collection", "grid_cell", "uri"]].head()